[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Cascades and Deletes &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/college.db` as the notebook's Setup did, and makes what its first
worked example made: the five advisors, Fall 2026 with sections 41 to 50, Zoe Nakamura, and three
Fall 2026 enrollments for every student. Run it first. The tasks do not depend on one another, and
the last cell removes the scratch folder.


In [1]:
import logging
import shutil
import warnings
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import (CheckConstraint, ForeignKey, MetaData, String, UniqueConstraint, create_engine, delete, event, func,
                        insert, select)
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, sessionmaker
from sqlalchemy.orm.exc import StaleDataError
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Advisor(Base):
    __tablename__ = "advisors"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))

    students: Mapped[list["Student"]] = relationship(back_populates="advisor", order_by="Student.id")

    def __repr__(self):
        return f"Advisor({self.name!r})"


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]
    advisor_id: Mapped[int | None] = mapped_column(ForeignKey("advisors.id"))

    advisor: Mapped[Advisor | None] = relationship(back_populates="students")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id",
                                                           cascade="all, delete-orphan")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id",
                                                     cascade="all, delete-orphan", passive_deletes=True)

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id", ondelete="CASCADE"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id",
                                                           cascade="all, delete-orphan", passive_deletes=True)

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id", ondelete="CASCADE"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


def build_college(engine):
    """Create the college's tables from the classes, load the lists above into them, and count their rows."""
    Base.metadata.create_all(engine)
    rows = {
        Course: [{"code": code, "title": title, "department": department, "credits": credits}
                 for code, title, department, credits in COURSES],
        Student: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                  for name, email, program, started in STUDENTS],
        Term: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        Section: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        Enrollment: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                     for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for cls, values in rows.items():
            conn.execute(insert(cls), values)
        return {cls.__tablename__: conn.execute(select(func.count()).select_from(cls)).scalar_one() for cls in rows}


engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))

SessionLocal = sessionmaker(engine)


ADVISORS = {"Biology": "Miriam Hale", "Computer Science": "Samuel Osei", "Mathematics": "Lucia Ferrante",
            "Psychology": "Anders Lund", "History": "Nadia Rahman"}

with SessionLocal.begin() as session:
    advisors = {program: Advisor(name=name) for program, name in ADVISORS.items()}
    for student in session.scalars(select(Student)):
        student.advisor = advisors[student.program]                 # one advisor for every program
    fall = Term(name="Fall 2026", starts_on=date(2026, 8, 24))
    session.add(fall)
    for course in session.scalars(select(Course).order_by(Course.id)):
        fall.sections.append(Section(course=course, capacity=30))   # sections 41 to 50
    session.add(Student(name="Zoe Nakamura", email="znakamura@college.edu", program="Computer Science",
                        started_on=date(2026, 8, 24), advisor=advisors["Computer Science"]))
    for student in session.scalars(select(Student).order_by(Student.id)):
        for step in (0, 3, 6):                                      # three courses each, as in every term
            session.add(Enrollment(student=student, section=fall.sections[(student.id + step) % 10]))


def counts():
    """The number of rows in every table this notebook deletes from."""
    with SessionLocal() as session:
        return {cls.__tablename__: session.scalar(select(func.count()).select_from(cls))
                for cls in (Advisor, Student, Term, Section, Enrollment)}


print(counts())


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}
{'advisors': 5, 'students': 26, 'terms': 5, 'sections': 50, 'enrollments': 306}


**1.** Anders Lund leaves, and the default rule runs.


In [2]:
engine.echo = True
with SessionLocal() as session:
    anders = session.scalars(select(Advisor).where(Advisor.name == "Anders Lund")).one()
    session.delete(anders)
    session.commit()
engine.echo = False

with SessionLocal() as session:
    print(session.scalars(select(Student.name).where(Student.advisor_id.is_(None)).order_by(Student.id)).all())


    BEGIN (implicit)
    SELECT advisors.id, advisors.name
    FROM advisors
    WHERE advisors.name = ?
    values: ('Anders Lund',)
    SELECT students.id AS students_id, students.name AS students_name, students.email AS students_email, students.program AS students_program, students.started_on AS students_started_on, students.advisor_id AS students_advisor_id
    FROM students
    WHERE ? = students.advisor_id ORDER BY students.id
    values: (4,)
    UPDATE students SET advisor_id=? WHERE students.id = ?
    values: [(None, 4), (None, 9), (None, 14), (None, 19), (None, 24)]
    DELETE FROM advisors WHERE advisors.id = ?
    values: (4,)
    COMMIT
['Daniel Kim', 'Isabel Costa', 'Noah Andersen', 'Sam Ito', 'Yara Haddad']


The session loaded the five Psychology students, set their `advisor_id` to `NULL` in one `UPDATE`,
and deleted the advisor after that.


**2.** A student admitted and deleted, with the enrollments.


In [3]:
with SessionLocal.begin() as session:
    psychology, statistics = session.get(Section, 49), session.get(Section, 50)            # both Fall 2026
    session.add(Student(name="Priya Shah", email="pshah@college.edu", program="Psychology", started_on=date(2026, 8, 24),
                        enrollments=[Enrollment(section=psychology), Enrollment(section=statistics)]))

with SessionLocal() as session:
    print("enrollments before:", session.scalar(select(func.count()).select_from(Enrollment)))

engine.echo = True
with SessionLocal() as session:
    session.delete(session.scalars(select(Student).where(Student.name == "Priya Shah")).one())
    session.commit()
engine.echo = False

with SessionLocal() as session:
    print("enrollments after: ", session.scalar(select(func.count()).select_from(Enrollment)))


enrollments before: 308
    BEGIN (implicit)
    SELECT students.id, students.name, students.email, students.program, students.started_on, students.advisor_id
    FROM students
    WHERE students.name = ?
    values: ('Priya Shah',)
    SELECT enrollments.student_id AS enrollments_student_id, enrollments.section_id AS enrollments_section_id, enrollments.status AS enrollments_status, enrollments.grade AS enrollments_grade
    FROM enrollments
    WHERE ? = enrollments.student_id ORDER BY enrollments.section_id
    values: (27,)
    DELETE FROM enrollments WHERE enrollments.student_id = ? AND enrollments.section_id = ?
    values: [(27, 49), (27, 50)]
    DELETE FROM students WHERE students.id = ?
    values: (27,)
    COMMIT
enrollments after:  306


`Student.enrollments` has `cascade="all, delete-orphan"`, so the session loaded Priya Shah's two
enrollments, deleted them, and deleted the student last. The first block looks both sections up
before it builds anything, since a query between the new objects would flush them half-built, as
the **Relationships** notebook showed.


**3.** An enrollment taken out of its collection.


In [4]:
engine.echo = True
with SessionLocal() as session:
    daniel = session.get(Student, 4)
    history = next(enrollment for enrollment in daniel.enrollments if enrollment.section_id == 48)   # World History
    daniel.enrollments.remove(history)
    print("Fall 2026:", [enrollment.section_id for enrollment in daniel.enrollments if enrollment.section_id > 40])
    session.commit()
engine.echo = False


    BEGIN (implicit)
    SELECT students.id AS students_id, students.name AS students_name, students.email AS students_email, students.program AS students_program, students.started_on AS students_started_on, students.advisor_id AS students_advisor_id
    FROM students
    WHERE students.id = ?
    values: (4,)
    SELECT enrollments.student_id AS enrollments_student_id, enrollments.section_id AS enrollments_section_id, enrollments.status AS enrollments_status, enrollments.grade AS enrollments_grade
    FROM enrollments
    WHERE ? = enrollments.student_id ORDER BY enrollments.section_id
    values: (4,)
Fall 2026: [41, 45]
    DELETE FROM enrollments WHERE enrollments.student_id = ? AND enrollments.section_id = ?
    values: (4, 48)
    COMMIT


The collection lost the enrollment at once, and `delete-orphan` turned that into a `DELETE` at the
commit. Fall 2026's sections are 41 to 50, so the test on `section_id` picks out that term.


**4.** A section canceled, and the database deletes its enrollments.


In [5]:
IN_DATA_STRUCTURES = select(func.count()).where(Enrollment.section_id == 46)          # Data Structures, Fall 2026

with SessionLocal() as session:
    print("before:", session.scalar(IN_DATA_STRUCTURES))

engine.echo = True
with SessionLocal() as session:
    session.delete(session.get(Section, 46))
    session.commit()
engine.echo = False

with SessionLocal() as session:
    print("after: ", session.scalar(IN_DATA_STRUCTURES))


before: 8
    BEGIN (implicit)
    SELECT sections.id AS sections_id, sections.course_id AS sections_course_id, sections.term_id AS sections_term_id, sections.capacity AS sections_capacity
    FROM sections
    WHERE sections.id = ?
    values: (46,)
    DELETE FROM sections WHERE sections.id = ?
    values: (46,)
    COMMIT
after:  0


One `DELETE` for the section, and the database deleted the eight enrollments as part of it.


**5.** One bulk `delete()` for a department's courses.


In [6]:
MATHEMATICS_FALL = (
    select(Section.id)
    .join(Section.term)
    .join(Section.course)
    .where(Term.name == "Fall 2026", Course.department == "Mathematics")
)
with SessionLocal() as session:
    result = session.execute(delete(Enrollment).where(Enrollment.section_id.in_(MATHEMATICS_FALL)))
    session.commit()
print(result.rowcount, "enrollments deleted")


23 enrollments deleted


The Mathematics department teaches three of the courses, Calculus I, Calculus II and Statistics, so
the subquery found three sections, and the one statement deleted their enrollments.


**6.** A section deleted with its enrollments already loaded.


In [7]:
from sqlalchemy.orm import selectinload

engine.echo = True
with SessionLocal() as session:
    composition = session.scalars(
        select(Section).where(Section.id == 47).options(selectinload(Section.enrollments))   # Composition, Fall 2026
    ).one()
    session.delete(composition)
    session.commit()
engine.echo = False


    BEGIN (implicit)
    SELECT sections.id, sections.course_id, sections.term_id, sections.capacity
    FROM sections
    WHERE sections.id = ?
    values: (47,)
    SELECT enrollments.section_id AS enrollments_section_id, enrollments.student_id AS enrollments_student_id, enrollments.status AS enrollments_status, enrollments.grade AS enrollments_grade
    FROM enrollments
    WHERE enrollments.section_id IN (?) ORDER BY enrollments.student_id
    values: (47,)
    DELETE FROM enrollments WHERE enrollments.student_id = ? AND enrollments.section_id = ?
    values: [(3, 47), (6, 47), (10, 47), (13, 47), (16, 47), (20, 47), (23, 47), (26, 47)]
    DELETE FROM sections WHERE sections.id = ?
    values: (47,)
    COMMIT


The session did, this time. `passive_deletes=True` keeps the session from loading children just to
delete them, and these were loaded already, so the session deleted all eight with a `DELETE` of its
own before the section's, and the database's rule had nothing left to delete.

Last, remove the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Cascades and Deletes](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/15-cascades-and-deletes.ipynb)  &nbsp;&middot;&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
